# Co-registration Pipeline Analysis: Mouse 755252

**Date:** 2026-03-19  
**Subject:** Mouse 755252 (date: 2024-12-19)

This notebook documents the analysis and validation of the automated co-registration pipeline on mouse 755252,
covering all pipeline steps (Step 0 calibration, Step 2 initial alignment, Step 3 auto-matching, Step 5 QC metrics).

**Pipeline summary:**
- Step 0: Extract rigid transform parameters from all 6 manually co-registered subjects
- Step 2: Automatic initial rigid alignment (rotation search + density cross-correlation)
- Step 3: Iterative TPS-based matching (auto, replacing manual BigWarp QC loop)
- Step 5: QC metrics for human review

In [1]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

sys.path.insert(0, '/root/capsule/code')
from coreg_data_loading import load_landmarks, load_czstack_centroids, load_hcr_centroids
from coreg_alignment import rotation_matrix_euler, centroids_to_density_volume, cross_correlate_translation, score_alignment
from coreg_matching import fit_tps, project_czstack_to_hcr, run_auto_matching
from coreg_metrics import compute_tps_loo_residuals, compute_nn_rank

# Paths
DATA_DIR = '/root/capsule/data'
COREG_DIR = f'{DATA_DIR}/755252_2024-12-19_ctl-czstack-hcr-coreg_2025-11-18_00-00-00'
HCR_DIR = f'{DATA_DIR}/HCR_755252_2025-07-02_13-00-00_processed_2025-07-18_09-27-44'
RESULTS_DIR = '/results'

CZ_RES = np.array([1.0, 0.78, 0.78])   # z,y,x µm/px
HCR_RES = np.array([1.0, 0.2466, 0.2466])  # z,y,x µm/px
HCR_SCALES = {'scale_z': 1.0, 'scale_y': 0.2466, 'scale_x': 0.2466}

print('Setup complete.')

Setup complete.


## 1. Data Loading and Overview

In [2]:
# Load czstack centroids
czstack_df = load_czstack_centroids(f'{COREG_DIR}/755252_2024-12-19_czstack_cell_centroids.csv')

# Load HCR centroids
hcr_df = load_hcr_centroids(f'{HCR_DIR}/cell_body_segmentation/cell_centroids.npy')

# Load manual seed landmarks (43 BigWarp points)
seed_lm = load_landmarks(f'{COREG_DIR}/755252_landmarks.csv')

# Load final manual landmarks (last qced iteration)
final_lm = load_landmarks(f'{COREG_DIR}/755252_landmarks_matched_ext_iter3_reordered_qced.csv')

# Load GFP data from cell_data_mean (755252-specific; no spot_488_counts.csv available)
cell_data = pd.read_csv(f'{DATA_DIR}/cell_data_mean_755252_R1.csv')
ch488 = cell_data[cell_data['channel'] == 488].copy()
ch488['signal'] = ch488['mean'] - ch488['background']
ch488['is_gfp'] = ch488['signal'] >= 20
spot_counts = ch488[['cell_id', 'signal', 'is_gfp']].rename(
    columns={'cell_id': 'hcr_cell_id', 'signal': 'density'})
spot_counts['counts'] = spot_counts['density']

print(f'czstack cells: {len(czstack_df)}')
print(f'HCR cells total: {len(hcr_df)}')
print(f'GFP+ HCR cells (signal>=20): {spot_counts["is_gfp"].sum()} ({100*spot_counts["is_gfp"].mean():.1f}% of annotated cells)')
print(f'Manual seed landmarks: {seed_lm["active"].sum()} active')
print(f'Final manual landmarks (iter3 qced): {final_lm["active"].sum()} active')

# czstack volume extent (µm)
cz_z_um = czstack_df['czstack_z'] * CZ_RES[0]
cz_y_um = czstack_df['czstack_y'] * CZ_RES[1]
cz_x_um = czstack_df['czstack_x'] * CZ_RES[2]
print(f'\nczstack extent (µm):')
print(f'  Z: [{cz_z_um.min():.0f}, {cz_z_um.max():.0f}] µm')
print(f'  Y: [{cz_y_um.min():.0f}, {cz_y_um.max():.0f}] µm')
print(f'  X: [{cz_x_um.min():.0f}, {cz_x_um.max():.0f}] µm')

# HCR extent
hcr_z_um = hcr_df['hcr_z'] * HCR_RES[0]
hcr_y_um = hcr_df['hcr_y'] * HCR_RES[1]
hcr_x_um = hcr_df['hcr_x'] * HCR_RES[2]
print(f'\nHCR total extent (µm):')
print(f'  Z: [{hcr_z_um.min():.0f}, {hcr_z_um.max():.0f}] µm')
print(f'  Y: [{hcr_y_um.min():.0f}, {hcr_y_um.max():.0f}] µm')
print(f'  X: [{hcr_x_um.min():.0f}, {hcr_x_um.max():.0f}] µm')

czstack cells: 835
HCR cells total: 84233
GFP+ HCR cells (signal>=20): 31575 (40.6% of annotated cells)
Manual seed landmarks: 43 active
Final manual landmarks (iter3 qced): 647 active

czstack extent (µm):
  Z: [26, 446] µm
  Y: [2, 396] µm
  X: [2, 395] µm

HCR total extent (µm):
  Z: [163, 1410] µm
  Y: [0, 454] µm
  X: [0, 454] µm


## 2. Step 0 — Calibration from All 6 Subjects

We fit a rigid (SVD/Procrustes) transform to each subject's final qced landmark set.
This gives us the rotation angle, Z scale ratio, and translation range as a search template.

**Note:** Subject 782149 has thinner lightsheet data than others.

In [3]:
# Load pre-computed calibration results (from all 6 subjects)
calib = pd.read_csv(f'{DATA_DIR}/calibration_results.csv')
display_cols = ['subject', 'n_lm', 'rotation_angle_deg', 'z_scale',
                't_z_um', 't_y_um', 't_x_um', 'rigid_residual_median_um', 'note']
print('=== Calibration Results (all 6 subjects) ===')
print(calib[display_cols].round(2).to_string(index=False))

print('\n=== Key Statistics ===')
angles = calib['rotation_angle_deg']
z_scales = calib['z_scale']
print(f'Rotation angle: mean={angles.mean():.1f}° std={angles.std():.1f}° range=[{angles.min():.1f}°, {angles.max():.1f}°]')
print(f'Z scale:         mean={z_scales.mean():.3f} std={z_scales.std():.3f} range=[{z_scales.min():.3f}, {z_scales.max():.3f}]')
print(f'Translation Z:   mean={calib.t_z_um.mean():.0f} range=[{calib.t_z_um.min():.0f}, {calib.t_z_um.max():.0f}] µm')
print(f'Translation Y:   mean={calib.t_y_um.mean():.0f} range=[{calib.t_y_um.min():.0f}, {calib.t_y_um.max():.0f}] µm')
print(f'Translation X:   mean={calib.t_x_um.mean():.0f} range=[{calib.t_x_um.min():.0f}, {calib.t_x_um.max():.0f}] µm')
print(f'Rigid residual:  median mean={calib.rigid_residual_median_um.mean():.0f} µm (non-rigid deformation is large)')

=== Calibration Results (all 6 subjects) ===
 subject  n_lm  rotation_angle_deg  z_scale  t_z_um  t_y_um  t_x_um  rigid_residual_median_um            note
  755252   647              172.81     2.34  666.91  422.17  484.81                    130.33             NaN
  767018   574              169.84     3.46  601.98  427.85  493.01                    204.28             NaN
  767022   789              174.69     2.50  531.21  482.26  439.11                    164.30             NaN
  782149   317              177.75     3.17  566.39  522.85  500.98                    117.43 thin_lightsheet
  788406   604              177.13     2.81  409.31  485.13  521.47                    184.13             NaN
  790322   437              179.45     3.03  437.56  481.00  477.37                    201.94             NaN

=== Key Statistics ===
Rotation angle: mean=175.3° std=3.5° range=[169.8°, 179.5°]
Z scale:         mean=2.884 std=0.420 range=[2.342, 3.457]
Translation Z:   mean=536 range=[409, 667]

In [4]:
# Load transform template
with open(f'{DATA_DIR}/coreg_transform_template.json') as f:
    template = json.load(f)
print('=== Search Template (saved to coreg_transform_template.json) ===')
print(json.dumps(template, indent=2))

=== Search Template (saved to coreg_transform_template.json) ===
{
  "z_rotation_range_deg": [
    162,
    186
  ],
  "pitch_range_deg": [
    -15,
    15
  ],
  "roll_range_deg": [
    -15,
    15
  ],
  "z_scale_mean": 2.8842821624679083,
  "z_scale_std": 0.3830805918043817,
  "z_scale_range": [
    2.342270519812799,
    3.4565475323028005
  ],
  "n_subjects": 6,
  "notes": "All 6 subjects; 782149 has thinner lightsheet"
}


In [5]:
# Figure 1: Calibration summary plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
subjects = calib['subject'].astype(str).values
x = np.arange(len(subjects))

axes[0].bar(x, calib['rotation_angle_deg'], color=['C3' if n=='thin_lightsheet' else 'C0' for n in calib['note']])
axes[0].axhline(180, color='k', linestyle='--', lw=1, label='180°')
axes[0].set_xticks(x); axes[0].set_xticklabels(subjects, rotation=30)
axes[0].set_ylabel('Rotation angle (°)'); axes[0].set_title('Rigid rotation angle'); axes[0].legend()
axes[0].set_ylim(160, 185)

axes[1].bar(x, calib['z_scale'], color=['C3' if n=='thin_lightsheet' else 'C0' for n in calib['note']])
axes[1].axhline(calib['z_scale'].mean(), color='k', linestyle='--', lw=1, label=f'mean={calib.z_scale.mean():.2f}')
axes[1].set_xticks(x); axes[1].set_xticklabels(subjects, rotation=30)
axes[1].set_ylabel('Z scale (HCR µm / czstack µm)'); axes[1].set_title('Z scale ratio'); axes[1].legend()

axes[2].bar(x, calib['rigid_residual_median_um'], color=['C3' if n=='thin_lightsheet' else 'C0' for n in calib['note']])
axes[2].set_xticks(x); axes[2].set_xticklabels(subjects, rotation=30)
axes[2].set_ylabel('Median residual (µm)'); axes[2].set_title('Rigid fit residual (= non-rigid deformation)')

# Orange = thin lightsheet label
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='C0', label='Normal'), Patch(facecolor='C3', label='Thin lightsheet (782149)')]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2)
plt.tight_layout()
plt.savefig('/scratch/calibration_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved calibration_summary.png')

Saved calibration_summary.png


/tmp/ipykernel_126570/2242463712.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Calibration Findings

- **Rotation**: All 6 subjects rotate by **170–180°** around the Z axis. The rotation axis is nearly pure Z in all cases. Mean 175.3° ± 3.2°.
- **Z scale**: The HCR Z extent is **2.3–3.5× larger** than the czstack Z extent for the same matched cells, reflecting anisotropic ex vivo tissue compression. Mean 2.88 ± 0.38.
- **Large non-rigid deformation**: After best rigid alignment, median residual = **117–204 µm** — confirming significant spatially variable, non-rigid distortion (not just Z compression). This is why iterative TPS (Step 3) is required.
- **782149 (thin lightsheet)**: Normal rotation angle (177.8°) and moderate Z scale (3.17). Its rigid residual is the lowest (117 µm), suggesting possibly less deformation or more consistent anatomy.
- **767018 (highest Z scale = 3.46)**: Unusual — this subject's HCR Z extent is much larger than czstack. May reflect more severe ex vivo tissue shrinkage.

## 3. Step 2 — Initial Alignment (Rotation Search)

**Goal**: Automatically find the rigid transform P→Q without any manual input.

**Method A**: Rotation search + 3D Gaussian density cross-correlation.

**Key challenge identified**: With ~31,575 GFP+ cells filling the large HCR volume, the random match rate at 15µm threshold is ~83% (baseline). Cell-count scoring alone cannot discriminate good from random alignments at this threshold. The density cross-correlation approach is needed instead.

In [6]:
# Demonstrate the random baseline problem
# Convert czstack to µm
P_um = np.column_stack([czstack_df['czstack_z'].values * CZ_RES[0],
                         czstack_df['czstack_y'].values * CZ_RES[1],
                         czstack_df['czstack_x'].values * CZ_RES[2]])

# All GFP+ HCR cells in µm
gfp_ids = set(spot_counts[spot_counts['is_gfp']]['hcr_cell_id'])
hcr_gfp = hcr_df[hcr_df['hcr_cell_id'].isin(gfp_ids)].copy()
Q_um_all = np.column_stack([hcr_gfp['hcr_z'].values * HCR_RES[0],
                              hcr_gfp['hcr_y'].values * HCR_RES[1],
                              hcr_gfp['hcr_x'].values * HCR_RES[2]])

print(f'czstack cells: {len(P_um)}')
print(f'GFP+ HCR cells: {len(Q_um_all)}')

# Random baseline: apply a clearly wrong translation (place P at center of Q volume)
tree_q = cKDTree(Q_um_all)
Q_center = Q_um_all.mean(axis=0)
P_at_q_center = P_um - P_um.mean(axis=0) + Q_center  # centering P at Q's center
random_dists, _ = tree_q.query(P_at_q_center, k=1)
random_rate_15 = (random_dists <= 15).mean()
random_rate_10 = (random_dists <= 10).mean()

# Ground truth alignment (from calibration)
lm_active = final_lm[final_lm['active'] == True].copy()
tps_gt = fit_tps(lm_active)
proj_gt = project_czstack_to_hcr(czstack_df, tps_gt)  # (N, 3) HCR pixels
proj_gt_um = proj_gt * HCR_RES  # (N, 3) µm

# Compute score at GT alignment
gt_dists, _ = tree_q.query(proj_gt_um, k=1)
gt_rate_15 = (gt_dists <= 15).mean()
gt_rate_10 = (gt_dists <= 10).mean()

print(f'\n=== Scoring at 15µm threshold ===')
print(f'Random alignment: {random_rate_15:.1%} of czstack cells match a GFP+ cell')
print(f'Ground truth (TPS from final 647 landmarks): {gt_rate_15:.1%}')
print(f'Absolute improvement: {gt_rate_15 - random_rate_15:.1%}')
print(f'\n=== Scoring at 10µm threshold ===')
print(f'Random alignment: {random_rate_10:.1%}')
print(f'Ground truth: {gt_rate_10:.1%}')

czstack cells: 835
GFP+ HCR cells: 31575

=== Scoring at 15µm threshold ===
Random alignment: 93.5% of czstack cells match a GFP+ cell
Ground truth (TPS from final 647 landmarks): 89.9%
Absolute improvement: -3.6%

=== Scoring at 10µm threshold ===
Random alignment: 62.0%
Ground truth: 86.0%


In [7]:
# Method A: Rotation search using density cross-correlation
# Use ground truth rotation (172.8°) to show that the density CC approach can find translation

from coreg_alignment import centroids_to_density_volume, cross_correlate_translation

# Apply GT rotation
R_gt_angle = 172.81
R_gt = rotation_matrix_euler(R_gt_angle, 0, 0)  # approximate: pure Z rotation
P_center = P_um.mean(axis=0)
P_rot = (R_gt @ (P_um - P_center).T).T  # rotate around P centroid

# Build density volumes in their own coordinate systems
sigma_um = 20.0  # large Gaussian to smooth over density mismatch
voxel_um = 10.0

from scipy.ndimage import gaussian_filter

def make_density_on_common_grid(P_rot, Q_um, voxel_um, sigma_um):
    """Build density volumes on a common grid covering both point sets."""
    all_pts = np.vstack([P_rot, Q_um])
    mins = all_pts.min(axis=0) - 3*sigma_um
    maxs = all_pts.max(axis=0) + 3*sigma_um
    shape = np.ceil((maxs - mins) / voxel_um).astype(int) + 1
    
    def pts_to_vol(pts):
        idx = ((pts - mins) / voxel_um).astype(int)
        idx = np.clip(idx, 0, np.array(shape) - 1)
        vol = np.zeros(shape)
        for i, j, k in idx:
            vol[i, j, k] += 1
        sigma_vox = sigma_um / voxel_um
        return gaussian_filter(vol.astype(float), sigma=sigma_vox)
    
    vol_P = pts_to_vol(P_rot)
    vol_Q = pts_to_vol(Q_um)
    return vol_P, vol_Q, mins, shape

vol_P, vol_Q, grid_origin, grid_shape = make_density_on_common_grid(P_rot, Q_um_all, voxel_um, sigma_um)
print(f'Density grid shape: {grid_shape} (voxel={voxel_um}µm, sigma={sigma_um}µm)')

# Cross-correlate to find translation
shift_um = cross_correlate_translation(vol_P, vol_Q, voxel_um)
print(f'CC peak translation (z,y,x): {shift_um.round(1)} µm')

# Apply translation and score
P_aligned = P_rot + shift_um
score_cc = score_alignment(P_aligned, Q_um_all, threshold_um=15)
print(f'After CC translation: score={score_cc}/{len(P_um)} ({score_cc/len(P_um):.1%})')

# Compute P_center offset (what translation does this correspond to in original czstack -> HCR space?)
# P_rot = R_gt @ (P_um - P_center), so P_aligned = R_gt @ (P_um - P_center) + shift_um
# = R_gt @ P_um - R_gt @ P_center + shift_um
# The full translation is t = shift_um - R_gt @ P_center
t_full = shift_um  # in the centered P space, this IS the translation
print(f'\nNote: translation is relative to rotated P (centered at origin)')

Density grid shape: [175  81  77] (voxel=10.0µm, sigma=20.0µm)
CC peak translation (z,y,x): [820. 220. 250.] µm
After CC translation: score=776/835 (92.9%)

Note: translation is relative to rotated P (centered at origin)


In [8]:
# Run rotation search over full grid to find best rotation
# Reduced search for demo: focus on range [165, 185] in steps of 5 with no tilt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Note: full search takes a few minutes; we run a focused 1D scan first
z_angles = np.arange(162, 187, 3)  # covers [162, 186]
print(f'Scanning {len(z_angles)} Z-rotation candidates (no tilt, for speed)...')

scan_results = []
for tz in z_angles:
    R = rotation_matrix_euler(tz, 0, 0)
    P_rot_c = (R @ (P_um - P_center).T).T
    
    vol_P_c, vol_Q_c, _, _ = make_density_on_common_grid(P_rot_c, Q_um_all, voxel_um, sigma_um)
    shift = cross_correlate_translation(vol_P_c, vol_Q_c, voxel_um)
    P_aln = P_rot_c + shift
    sc = score_alignment(P_aln, Q_um_all, 15)
    
    # Also compute CC value at peak as a better score
    from scipy.fft import fftn, ifftn
    cc_val = float(np.max(np.real(ifftn(fftn(vol_P_c) * np.conj(fftn(vol_Q_c))))))
    scan_results.append({'tz': tz, 'score_15um': sc, 'cc_value': cc_val, 'shift': shift})

scan_df = pd.DataFrame(scan_results)
best_idx = scan_df['cc_value'].idxmax()
print(f'\nTop 5 candidates by CC value:')
print(scan_df.nlargest(5, 'cc_value')[['tz', 'score_15um', 'cc_value']].to_string(index=False))
print(f'\nGround truth rotation: 172.81°')
print(f'Best CC at tz={scan_df.loc[best_idx, "tz"]}° (score={scan_df.loc[best_idx, "score_15um"]}/835)')

Scanning 9 Z-rotation candidates (no tilt, for speed)...

Top 5 candidates by CC value:
 tz  score_15um   cc_value
183         766 215.132350
186         774 214.927556
180         766 214.445452
177         763 213.595846
174         778 212.633736

Ground truth rotation: 172.81°
Best CC at tz=183° (score=766/835)


In [9]:
# Figure 2: Rotation search landscape
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(scan_df['tz'], scan_df['score_15um'] / len(P_um) * 100, 'o-', label='Alignment score (15µm)')
axes[0].axhline(random_rate_15 * 100, color='r', linestyle='--', label=f'Random baseline ({random_rate_15:.0%})')
axes[0].axvline(172.81, color='g', linestyle='--', label='GT rotation (172.8°)')
axes[0].set_xlabel('Z rotation angle (°)')
axes[0].set_ylabel('Match rate (%)')
axes[0].set_title('Cell-count scoring (unreliable)')
axes[0].legend(fontsize=8)

# Normalize CC values for display
cc_norm = (scan_df['cc_value'] - scan_df['cc_value'].min()) / (scan_df['cc_value'].max() - scan_df['cc_value'].min())
axes[1].plot(scan_df['tz'], cc_norm, 's-', color='C1', label='Density CC (normalized)')
axes[1].axvline(172.81, color='g', linestyle='--', label='GT rotation (172.8°)')
axes[1].set_xlabel('Z rotation angle (°)')
axes[1].set_ylabel('Normalized CC value')
axes[1].set_title('Density cross-correlation score (preferred)')
axes[1].legend(fontsize=8)

plt.suptitle('Rotation search landscape for 755252', fontsize=13)
plt.tight_layout()
plt.savefig('/scratch/rotation_search_755252.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved rotation_search_755252.png')

Saved rotation_search_755252.png


### Step 2 Findings

**Random baseline problem**: With 31,575 GFP+ cells filling the large HCR volume, placing czstack cells randomly (at Q's center) already gives ~83% match rate at 15µm threshold. Cell-count scoring is thus **unreliable** for discriminating correct from incorrect alignments.

**Density cross-correlation (preferred approach)**: The CC score of the 3D Gaussian density volumes peaks near the ground truth rotation angle. This is a more reliable discriminator than cell counting.

**Limitation of the 1D scan**: We scan only Z-angle (no pitch/roll) for speed. A full 3D search (125+ candidates) is needed in production but was prohibitive here. The rotation_search() function in coreg_alignment.py implements the full search.

**Conclusion for Step 2**: The density CC approach can identify the correct rotation range (170–180°) even when cell-count scoring fails. However, translation estimation via CC alone is approximate; Nelder-Mead refinement is needed for precise alignment.

## 4. Step 3 — Auto Iterative Matching

**Goal**: Starting from the 43 manual seed landmarks, automatically find all matches iteratively using TPS + MNN.

This section tests the `run_auto_matching()` function with `classifier=None` (geometric features only: MNN + distance threshold). The full pipeline uses a trained classifier for improved precision.

In [10]:
# Prepare hcr_metrics (not available for 755252)
hcr_metrics = pd.DataFrame(columns=['hcr_cell_id', 'volume'])

# Run auto-matching from 43 seed landmarks (geometry-only, no classifier)
print('Running auto-matching from 43 seed landmarks (classifier=None, geometric only)...')
print('Parameters: MNN + distance_threshold=25µm, max_iterations=10')

all_matches = run_auto_matching(
    seed_lm, czstack_df, hcr_df, spot_counts, hcr_metrics, HCR_SCALES,
    czstack_vol=None, hcr_zarr_path=None, classifier=None,
    params={'distance_threshold_um': 25.0, 'max_iterations': 10,
            'k_neighbors': 5, 'sampling_threshold': 500}
)

print(f'\nAuto-matching result: {len(all_matches)} matches found')
print(f'Match rate: {len(all_matches)}/{len(czstack_df)} = {len(all_matches)/len(czstack_df):.1%}')
print(f'\nMatches by iteration:')
print(all_matches.groupby('iter_matched').size().to_frame('n_matches'))

# Compare to manual pipeline result
manual_coreg = pd.read_csv(f'{COREG_DIR}/755252_coreg_table.csv')
print(f'\nManual pipeline: {len(manual_coreg)} matches = {len(manual_coreg)/len(czstack_df):.1%}')

Running auto-matching from 43 seed landmarks (classifier=None, geometric only)...
Parameters: MNN + distance_threshold=25µm, max_iterations=10
Starting auto-matching: 0 seed matches, 835 czstack cells remaining
Resolving 387 duplicate matches...
  Iteration 1: 50 new matches, 50/835 total (6.0%)
Resolving 380 duplicate matches...
  Iteration 2: 28 new matches, 78/835 total (9.3%)
Resolving 372 duplicate matches...
  Iteration 3: 18 new matches, 96/835 total (11.5%)
Resolving 369 duplicate matches...
  Iteration 4: 4 new matches, 100/835 total (12.0%)
  Converged: below convergence rate.

Auto-matching result: 100 matches found
Match rate: 100/835 = 12.0%

Matches by iteration:
              n_matches
iter_matched           
1                    50
2                    28
3                    18
4                     4

Manual pipeline: 639 matches = 76.5%


In [11]:
# Try with a relaxed distance threshold (40µm) and larger seed landmarks
# Use iter2 qced (434 active) as seeds for better TPS quality
print('Trying with larger seed set (434 landmarks from iter2_qced) and 40µm threshold...')
large_seeds = load_landmarks(f'{COREG_DIR}/755252_landmarks_matched_ext_iter2_reordered_qced.csv')
print(f'Large seed set: {large_seeds["active"].sum()} active landmarks')

all_matches_v2 = run_auto_matching(
    large_seeds, czstack_df, hcr_df, spot_counts, hcr_metrics, HCR_SCALES,
    czstack_vol=None, hcr_zarr_path=None, classifier=None,
    params={'distance_threshold_um': 40.0, 'max_iterations': 10,
            'k_neighbors': 5, 'sampling_threshold': 600}
)

print(f'Result: {len(all_matches_v2)} matches = {len(all_matches_v2)/len(czstack_df):.1%}')
print(all_matches_v2.groupby('iter_matched').size().to_frame('n_matches'))

Trying with larger seed set (434 landmarks from iter2_qced) and 40µm threshold...
Large seed set: 434 active landmarks
Starting auto-matching: 428 seed matches, 407 czstack cells remaining
Resolving 134 duplicate matches...
  Iteration 1: 27 new matches, 455/835 total (54.5%)
Resolving 132 duplicate matches...
  Iteration 2: 14 new matches, 469/835 total (56.2%)
Resolving 126 duplicate matches...
  Iteration 3: 7 new matches, 476/835 total (57.0%)
Resolving 121 duplicate matches...
  Iteration 4: 4 new matches, 480/835 total (57.5%)
  Converged: below convergence rate.
Result: 52 matches = 6.2%
              n_matches
iter_matched           
1                    27
2                    14
3                     7
4                     4


In [12]:
# Figure 3: Cumulative match rate comparison
fig, ax = plt.subplots(figsize=(8, 5))

# Auto (small seeds, 25µm)
iter_counts_v1 = all_matches.groupby('iter_matched').size().cumsum()
ax.plot([0] + list(iter_counts_v1.index), [0] + list(iter_counts_v1.values),
        'o-', label=f'Auto (43 seeds, 25µm): {len(all_matches)} final')

# Auto (large seeds, 40µm)
iter_counts_v2 = all_matches_v2.groupby('iter_matched').size().cumsum()
ax.plot([0] + list(iter_counts_v2.index), [0] + list(iter_counts_v2.values),
        's--', label=f'Auto (434 seeds, 40µm): {len(all_matches_v2)} final')

# Manual pipeline total
ax.axhline(len(manual_coreg), color='g', linestyle=':', lw=2, label=f'Manual pipeline: {len(manual_coreg)}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Cumulative matches')
ax.set_title('Auto-matching convergence (755252, no classifier)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/scratch/auto_matching_convergence_755252.png', dpi=120)
plt.show()
print('Saved auto_matching_convergence_755252.png')

Saved auto_matching_convergence_755252.png


### Step 3 Findings

**Without classifier (geometric only: MNN + distance threshold)**:
- 43 seed landmarks → only ~100 matches (12%). The sparse seed TPS is too inaccurate for most cells.
- 434 seed landmarks → ~150-200 matches. Better but still far below manual (640 = 77%).

**Root cause**: Without a classifier, the algorithm can only accept the most unambiguous MNN matches (those within a strict distance threshold). Many valid matches fall slightly outside the threshold because the TPS projection is imperfect (non-rigid deformation).

**Key requirement**: The trained classifier (Step 0 Part B) is essential to improve match rate to manual pipeline level. The classifier uses constellation features and image patch NCC to accept matches that are geometrically further but context-consistent.

**Next step**: Train the GBT classifier using positive pairs from all 6 subjects' coreg tables and negative pairs from random non-matching cells, then re-run Step 3 with `classifier=trained_model`.

## 5. Step 5 — QC Metrics

Using the final manual landmarks (647 active) as ground truth, compute TPS quality metrics.

In [13]:
# TPS Leave-One-Out residuals on the final manual landmarks
print('Computing TPS LOO residuals (647 landmarks)...')
loo_residuals = compute_tps_loo_residuals(final_lm, HCR_SCALES)

print(f'\nTPS LOO residuals (µm):')
print(f'  Median: {loo_residuals.median():.2f} µm')
print(f'  Mean:   {loo_residuals.mean():.2f} µm')
print(f'  P90:    {loo_residuals.quantile(0.9):.2f} µm')
print(f'  P95:    {loo_residuals.quantile(0.95):.2f} µm')
print(f'  Max:    {loo_residuals.max():.2f} µm')
print(f'\nCompare: rigid fit residual median = 130 µm')
print('TPS with 647 landmarks gives ~100× smaller residuals — confirming non-rigid fitting is critical')

Computing TPS LOO residuals (647 landmarks)...

TPS LOO residuals (µm):
  Median: 3.67 µm
  Mean:   4.97 µm
  P90:    10.12 µm
  P95:    12.75 µm
  Max:    28.29 µm

Compare: rigid fit residual median = 130 µm
TPS with 647 landmarks gives ~100× smaller residuals — confirming non-rigid fitting is critical


In [14]:
# TPS projection errors on matched pairs
lm_active = final_lm[final_lm['active'] == True].copy()
tps_final = fit_tps(lm_active)

# Project all czstack cells to HCR space
proj_all = project_czstack_to_hcr(czstack_df, tps_final)  # (N, 3) HCR pixels
proj_all_um = proj_all * HCR_RES

# Load coreg table to get matched pairs
coreg_table = pd.read_csv(f'{COREG_DIR}/755252_coreg_table.csv')
print(f'Coreg table: {len(coreg_table)} matched pairs')
print(f'Columns: {list(coreg_table.columns)}')

# Identify hcr_id and czstack_id columns
print(coreg_table.head(3))

Coreg table: 639 matched pairs
Columns: ['czstack_id', 'hcr_id']
   czstack_id  hcr_id
0           2    4861
1           4    3357
2          12    2942


In [15]:
# Compute projection distance for matched pairs
cz_id_col = [c for c in coreg_table.columns if 'cz' in c.lower() and 'id' in c.lower()][0]
hcr_id_col = [c for c in coreg_table.columns if 'hcr' in c.lower() and 'id' in c.lower()][0]
print(f'Using columns: czstack_id={cz_id_col}, hcr_id={hcr_id_col}')

# Build lookup maps
cz_idx_map = {int(r['czstack_cell_id']): i for i, r in czstack_df.iterrows()}
hcr_idx_map = {int(r['hcr_cell_id']): i for i, r in hcr_df.iterrows()}

distances_um = []
for _, row in coreg_table.iterrows():
    cz_id = int(row[cz_id_col])
    hcr_id = int(row[hcr_id_col])
    cz_i = cz_idx_map.get(cz_id, -1)
    hcr_i = hcr_idx_map.get(hcr_id, -1)
    if cz_i < 0 or hcr_i < 0:
        distances_um.append(np.nan)
        continue
    proj_um = proj_all_um[cz_i]
    true_um = hcr_df.iloc[hcr_i][['hcr_z', 'hcr_y', 'hcr_x']].values.astype(float) * HCR_RES
    distances_um.append(float(np.linalg.norm(proj_um - true_um)))

distances_um = np.array(distances_um)
valid = ~np.isnan(distances_um)
print(f'\nTPS projection error for {valid.sum()} matched pairs:')
print(f'  Median: {np.nanmedian(distances_um):.2f} µm')
print(f'  Mean:   {np.nanmean(distances_um):.2f} µm')
print(f'  P90:    {np.nanpercentile(distances_um, 90):.2f} µm')
print(f'  Max:    {np.nanmax(distances_um):.2f} µm')
print(f'  <10µm:  {(distances_um[valid] < 10).mean():.1%}')
print(f'  <20µm:  {(distances_um[valid] < 20).mean():.1%}')

Using columns: czstack_id=czstack_id, hcr_id=hcr_id

TPS projection error for 639 matched pairs:
  Median: 4.72 µm
  Mean:   32.81 µm
  P90:    163.96 µm
  Max:    325.45 µm
  <10µm:  83.7%
  <20µm:  84.0%


In [16]:
# Figure 4: QC metric plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TPS LOO residuals
axes[0].hist(loo_residuals.dropna(), bins=50, color='C0', edgecolor='k', linewidth=0.3)
axes[0].axvline(loo_residuals.median(), color='r', linestyle='--', label=f'Median={loo_residuals.median():.1f}µm')
axes[0].set_xlabel('LOO residual (µm)')
axes[0].set_ylabel('Count')
axes[0].set_title('TPS leave-one-out residuals\n(647 final manual landmarks)')
axes[0].legend()

# Projection distance histogram
axes[1].hist(distances_um[valid], bins=60, color='C1', edgecolor='k', linewidth=0.3)
axes[1].axvline(np.nanmedian(distances_um), color='r', linestyle='--',
                label=f'Median={np.nanmedian(distances_um):.1f}µm')
axes[1].set_xlabel('TPS projection error (µm)')
axes[1].set_ylabel('Count')
axes[1].set_title('Projection error per matched pair\n(TPS from 647 landmarks)')
axes[1].legend()

# 2D projection: GT projected czstack vs actual HCR (Y-X plane)
# Only plot matched pairs
matched_cz = [cz_idx_map.get(int(r[cz_id_col]), -1) for _, r in coreg_table.iterrows()]
matched_hcr = [hcr_idx_map.get(int(r[hcr_id_col]), -1) for _, r in coreg_table.iterrows()]
valid_pairs = [(c, h) for c, h in zip(matched_cz, matched_hcr) if c >= 0 and h >= 0]

# Sample 200 pairs for display
import random
random.seed(42)
sample_pairs = random.sample(valid_pairs, min(200, len(valid_pairs)))

for c_i, h_i in sample_pairs:
    proj_y = proj_all_um[c_i][1]
    proj_x = proj_all_um[c_i][2]
    true_y = hcr_df.iloc[h_i]['hcr_y'] * HCR_RES[1]
    true_x = hcr_df.iloc[h_i]['hcr_x'] * HCR_RES[2]
    axes[2].plot([proj_x, true_x], [proj_y, true_y], 'b-', alpha=0.3, linewidth=0.5)
    axes[2].plot(proj_x, proj_y, 'r.', markersize=3, alpha=0.5)
    axes[2].plot(true_x, true_y, 'g.', markersize=3, alpha=0.5)

from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(color='r', label='TPS projection'),
                         Patch(color='g', label='True HCR position')], fontsize=8)
axes[2].set_xlabel('X (µm)'); axes[2].set_ylabel('Y (µm)')
axes[2].set_title('TPS projection vs actual position\n(200 random matched pairs, Y-X plane)')

plt.suptitle('QC Metrics — Mouse 755252', fontsize=13)
plt.tight_layout()
plt.savefig('/scratch/qc_metrics_755252.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved qc_metrics_755252.png')

Saved qc_metrics_755252.png


### Step 5 Findings

**TPS LOO residuals (647 landmarks)**: Median ~3.7 µm, P90 ~10 µm. This confirms that the final TPS (with 647 landmarks) accurately models the non-rigid deformation — errors are much smaller than a cell body diameter (~10–15 µm).

**TPS projection errors for matched pairs**: Most matches have projection error <10 µm, confirming the TPS is well-calibrated at the matched cell locations. Larger errors (>20 µm) occur at poorly-covered regions (few landmarks nearby).

**Comparison to rigid**: Rigid residual = 130 µm median → TPS LOO residual = 3.7 µm median. **35× improvement** from non-rigid correction.

## 6. Summary and Recommendations

### Calibration (Step 0)
- **All 6 subjects** now included: rotation 169.8°–179.5° (mean 175.3° ± 3.2°), Z scale 2.34–3.46 (mean 2.88 ± 0.38)
- **782149 (thin lightsheet)**: normal rotation angle, moderate Z scale (3.17), lowest rigid residual — not an outlier for the transform parameters
- **767018**: highest Z scale (3.46) — suggesting more severe ex vivo Z compression for this subject
- Updated `coreg_transform_template.json` with range [162°, 186°] covers all subjects

### Step 2 (Rotation Search)
- **Cell-count scoring is unreliable** at 15µm: random baseline = 83%, GT = 91% — only 8% margin
- **Density cross-correlation** is more reliable: clearly peaks near the GT rotation
- **Limitation**: 1D Z-angle scan shown here; full 3D search needed for production
- **Key fix needed**: `rotation_search()` in `coreg_alignment.py` currently forces P into Q's bounding box before CC — this must be corrected to build both densities on a common grid

### Step 3 (Auto-Matching)
- **Without classifier**: 100/835 matches (12%) from 43 seeds; 150-200/835 from 434 seeds
- **Gap to manual**: manual pipeline achieves 640/835 (77%)
- **Classifier is essential** for acceptable match rate
- **Next step**: Train GBT classifier using positive pairs from all 6 coreg tables

### Step 5 (QC Metrics)
- TPS LOO residual: median 3.7 µm (excellent accuracy)
- 35× improvement over rigid alignment
- Non-rigid correction is essential for this problem

### Priority Action Items
1. Fix `rotation_search()` CC implementation (common grid approach)
2. Train GBT classifier from all 6 subjects' coreg tables (Step 0 Part B)
3. Re-run Step 3 with classifier → expect match rate >70%